### Testing RAG Applications 📑

#### RAG Application
This application reads data about Model Context Protocol (MCP) server from internet, stores in vector stores, chunks the data with embedding and useful to answer the question about MCP while inferenced.

<img src="./img/RAG.png" width="500" height="400" style="display: block; margin: auto;">

In [9]:
! uv pip install -qU langchain-chroma

In [ ]:
# from langchain_ollama import OllamaEmbeddings
# from langchain_chroma import Chroma
# from langchain_community.document_loaders import WebBaseLoader
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# from typing import List
# from langchain.prompts import ChatPromptTemplate
# from langchain.schema import StrOutputParser
# from langchain.schema.runnable import RunnablePassthrough
# from langchain.schema.document import Document
# from langchain_ollama import ChatOllama

In [10]:
from typing import List

# Ollama integrations
from langchain_ollama import OllamaEmbeddings, ChatOllama

# Vector store
from langchain_chroma import Chroma

# Document loading
from langchain_community.document_loaders import WebBaseLoader

# Text splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Prompts
from langchain_core.prompts import ChatPromptTemplate

# Output parsing
from langchain_core.output_parsers import StrOutputParser

# Runnables (LCEL)
from langchain_core.runnables import RunnablePassthrough

# Document schema
from langchain_core.documents import Document

In [11]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model = "glm-4.7:cloud",
    temperature=0.5,
    max_tokens = 250
)

In [13]:
# Load data from Web
loader = WebBaseLoader("https://www.descope.com/learn/post/mcp")
data = loader.load()

# Split text into documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(data)

# Add text to vector db
embedding = OllamaEmbeddings(model="nomic-embed-text:latest")
vectordb = Chroma.from_documents(documents=splits, embedding=embedding)

# Create a retriever
retriever = vectordb.as_retriever()

def format_docs(docs: List[Document]) -> str:
    return "\n\n".join([d.page_content for d in docs])


template = """Answer the question based only on the following context:

    {context}
    
    Give a summary not the full detail

    Question: {question}
    """
prompt = ChatPromptTemplate.from_template(template)

# --- RECTIFIED LCEL CHAIN ---
# We use standard LCEL (retriever | format_docs) instead of a custom function 
# to avoid the deprecated 'get_relevant_documents' method entirely!
chain = {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()



#### Output of the LLM Application

In [14]:
response = chain.invoke("What is MCP")

print(response)

Based on the provided context, there is no information available to answer what MCP is. The text only repeats the heading "## Why this matters" without any accompanying details.


### Testing RAG Application with DeepEval
<img src="./img/RAGTesting.png" width="800" height="400" style="display: block; margin: auto;">

In [17]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

confidentAi = os.getenv("CONFIDENTAI")

In [18]:
import deepeval

deepeval.login(confidentAi)

🎉🥳 Congratulations! You've successfully logged in! 🙌

In [19]:
# !deepeval set-ollama deepseek-r1:8b
# !deepeval set-ollama gemma4:31b-cloud
!deepeval set-ollama --model gemma4:31b-cloud
# !deepeval set-ollama --model gpt-oss:120b-cloud 

🙌 Congratulations! You're now using a local Ollama model `gemma4:31b-cloud` for
all evals that require an LLM.


In [20]:
from deepeval.test_case import LLMTestCase
from deepeval.dataset import EvaluationDataset

test_case = LLMTestCase(
    input="What is MCP?",
    actual_output=chain.invoke("What is MCP"),
    expected_output="The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps"
)

dataset = EvaluationDataset()
dataset.add_test_case(test_case=test_case)

In [21]:
dataset

EvaluationDataset(test_cases=[LLMTestCase(input='What is MCP?', actual_output='Based on the provided context, there is no information explaining what MCP is. The text only repeats the heading "Why this matters" and an instruction to give a summary.', expected_output='The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps', context=None, retrieval_context=None, additional_metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, completion_time=None, multimodal=False, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None, custom_column_key_values=None)], goldens=[], _alias=None, _id=None, _multi_turn=False)

In [22]:
from deepeval.test_case import LLMTestCaseParams
from deepeval.metrics import GEval

concise_metrics = GEval(
    name = "Concise",
    criteria="Assess if the actual output remains concise while preserving all essential information.",
    
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ]
)

In [23]:
from deepeval.test_case import LLMTestCaseParams
from deepeval.metrics import GEval

completness_metrics = GEval(
    name = "Completeness",
    criteria="Assess whether the actual output retains all the key information from the input",
    
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ]
)

### Evaluation with GEval 

In [24]:
from deepeval.evaluate import evaluate
from deepeval.metrics import AnswerRelevancyMetric

evaluate(dataset.test_cases, metrics=[
    completness_metrics, 
    AnswerRelevancyMetric(),
    concise_metrics
])

✨ You're running DeepEval's latest Completeness [GEval] Metric! (using gpt-oss:120b-cloud (Local Model), 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-oss:120b-cloud (Local Model), strict=False,
async_mode=True)...

✨ You're running DeepEval's latest Concise [GEval] Metric! (using gpt-oss:120b-cloud (Local Model), strict=False, 
async_mode=True)...

/Users/vaibhavarde/Desktop/TestAutomation/TestAutomationSkills/llmEvaluation/.venv/lib/python3.13/site-packages/ric
h/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Completeness [GEval] (score: 0.8, threshold: 0.5, strict: False, evaluation model: gpt-oss:120b-cloud (Local Model), reason: The response extracts the key facts from the source – that there is no explanation of MCP, only a repeated heading and a summary instruction – and lists exactly those points. It does not add or omit information, thus retaining the complete information present in the input., error: None)
  - ✅ Answer Relevancy (score: 0.5, threshold: 0.5, strict: False, evaluation model: gpt-oss:120b-cloud (Local Model), reason: The score is 0.50 because the response included unrelated content about headings and summary instructions, which detracted from directly answering the question about MCP, preventing a higher relevance rating., error: None)
  - ✅ Concise [GEval] (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-oss:120b-cloud (Local Model), reason: The response succinctly identifies the essential information (that the context lacks a

⚠ WARNING: No hyperparameters logged.
» ]8;id=16754768;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=16754772;https://app.confident-ai.com/project/cmo7i7fxg000gqc16oqwdl9nq/test-runs/cmomvf305005csd16q8seqxps/regression-testing\https://app.confident-ai.com/project/cmo7i7fxg000gqc16oqwdl9nq/test-runs/cmomvf305005csd16q8seqxps/regression-testi]8;;\
]8;id=16754772;https://app.confident-ai.com/project/cmo7i7fxg000gqc16oqwdl9nq/test-runs/cmomvf305005csd16q8seqxps/regression-testing\ng]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Completeness [GEval]', threshold=0.5, success=True, score=0.8, reason='The response extracts the key facts from the source – that there is no explanation of MCP, only a repeated heading and a summary instruction – and lists exactly those points. It does not add or omit information, thus retaining the complete information present in the input.', strict_mode=False, evaluation_model='gpt-oss:120b-cloud (Local Model)', error=None, evaluation_cost=0.0, verbose_logs='Criteria:\nAssess whether the actual output retains all the key information from the input \n \nEvaluation Steps:\n[\n    "Extract the main facts, concepts, and details that constitute the key information from the input.",\n    "List the information presented in the actual output.",\n    "Check that every extracted key point is present in the output and is not altered or omitted.",\n    "Determine if the output fully retain